In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

qmnist = unpickle("/kaggle/input/qmnist-the-extended-mnist-dataset-120k-images/MNIST-120k")

In [ ]:
qmnist.keys()

In [ ]:
qmnist['data'].shape, qmnist['labels'].shape

In [ ]:
# Lets show one of the images
import matplotlib.pyplot as plt

image = qmnist['data'][0]
plt.imshow(image, cmap='gray')
plt.show()
print(qmnist['labels'][0])

In [ ]:
# In order to use it in torch, we need to create a Custom dataset

import torch
from torch.utils.data import Dataset, DataLoader

class TensorBatchDataset(Dataset):
    def __init__(self, images_tensor, labels_tensor):
        self.images = torch.tensor(images_tensor, dtype=torch.float32)
        self.images = self.images.unsqueeze(1)
        self.images /= 255
        self.labels = torch.tensor(labels_tensor, dtype=torch.long).squeeze()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [ ]:
dataset = TensorBatchDataset(qmnist['data'], qmnist['labels'])
image, label = dataset[0]
image.shape, torch.min(image), torch.max(image), label

In [ ]:
batch_size = 64
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
from torch.utils.data import random_split

train_ratio = 0.8
train_size = int(len(dataset)*train_ratio)
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=False)


In [ ]:
images, labels = next(iter(train_dataloader))
images.shape, labels.shape

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# Define the CNN architecture
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.sig = nn.Sigmoid()

        # Layer 1: Convolutional + ReLU + MaxPool
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        
        # Layer 2: Convolutional + ReLU + MaxPool
        self.conv2 = nn.Conv2d(4, 8, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        
        # Layer 3: Fully connected
        self.fc1 = nn.Linear(8 * 7 * 7, 128)  # 28x28 -> 14x14 -> 7x7 after two max pools
        self.fc2 = nn.Linear(128, 10)  # 10 classes for digits 0-9
        
    def forward(self, x):
        # Layer 1
        x = self.conv1(x)
        x = self.sig(x)
        x = self.pool1(x)
        
        # Layer 2
        x = self.conv2(x)
        x = self.sig(x)
        x = self.pool2(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Layer 3
        x = self.fc1(x)
        x = self.sig(x)
        x = self.fc2(x)
        
        return x

In [ ]:
model = SimpleCNN()
output = model(images)
output.shape

In [ ]:
model = SimpleCNN().cuda()
optimizer = optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
softmax = nn.Softmax(dim=1)

train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=False)

EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    
    train_loss = 0
    correct_count = 0
    for images, label in train_dataloader:
        images = images.cuda()
        label = label.cuda()
        
        outputs = model(images)
        loss = criterion(outputs, label)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        correct_count += torch.sum(torch.argmax(softmax(outputs), dim=1) == label)
        
    train_accuracy = correct_count / len(train_dataset)

    model.eval()
    test_loss = 0
    correct_count = 0
    for images, label in test_dataloader:
        images = images.cuda()
        label = label.cuda()
        with torch.no_grad():
            outputs = model(images)
        loss = criterion(outputs, label)
        test_loss += loss.item()
        correct_count += torch.sum(torch.argmax(softmax(outputs), dim=1) == label)

    test_accuracy = correct_count / len(test_dataset)
 
    print(f'Epoch {epoch+1}, TrainLoss: {train_loss:.4f}, TestLoss: {test_loss:.4f}'+
          f', Acc: {train_accuracy:.4f}, TestAcc: {test_accuracy:.4f}')

In [ ]:
import matplotlib.pyplot as plt

def plot_image_grid(image_array, image_size=2, value_lims=None):
    # Determine the grid dimensions
    rows = len(image_array)          # Number of rows in the 2D array
    cols = len(image_array[0])       # Number of columns in the 2D array

    # Create a figure with a grid of subplots
    fig, axes = plt.subplots(rows, cols, figsize=(cols * image_size, rows * image_size))

    # Flatten the axes array for easy iteration (in case it's 1D or 2D)
    # axes = axes.flatten() if rows > 1 and cols > 1 else [axes]

    # Iterate through the 2D array of images and plot them
    for i in range(rows):
        for j in range(cols):
            img = image_array[i][j] 
            if rows > 1:
                ax = axes[i][j]
            else:
                ax = axes[j]
            # Plot the image on the corresponding axis
            if value_lims is None:
                ax.imshow(img, cmap='gray')  # Use 'cmap' for grayscale images
            else:
                vmin, vmax = value_lims
                ax.imshow(img, cmap='gray', vmin=vmin, vmax=vmax)
            ax.axis('off')  # Turn off axis labels for a cleaner look

    # Adjust spacing between subplots to minimize wasted space
    plt.tight_layout(pad=0.5)

    # Show the plot
    plt.show()

def to_image(t):
    return t.cpu().squeeze(0).numpy()

In [ ]:
exp_dataloader = DataLoader(train_dataset, batch_size=1024, shuffle=False)
images, _ = next(iter(exp_dataloader))
images = images.to(device)
images.shape

In [ ]:
with torch.no_grad():
    x = model.conv1(images)
    x = model.sig(x)
    x = x.cpu()
x.shape

In [ ]:
mixed = torch.mean(x, dim=0)
mixed.shape

In [ ]:
weights = model.conv1.weight.cpu().detach()
plot_image_grid([
    [weights[0,0], weights[1,0]],
    [weights[2,0], weights[3,0]],
], image_size=2, value_lims=(-1, 1))

In [ ]:
plot_image_grid([
    [mixed[0], mixed[1]],
    [mixed[2], mixed[3]],
])

In [ ]:
with torch.no_grad():
    x = model.conv1(images.cuda())
    x = model.sig(x)
    x = model.pool1(x)

    x = model.conv2(x)
    x = model.sig(x).cpu()
print(model.conv2.weight.shape)
x.shape

In [ ]:
mixed = torch.mean(x, dim=0)
plot_image_grid([
    [mixed[0], mixed[1], mixed[2], mixed[3]],
    [mixed[4], mixed[5], mixed[6], mixed[7]],
])